# Reset ESPN silver tables

**Destructive.** Drops the ESPN silver tables (`nfl.teams`, `nfl.players`, `nfl.rosters`, `nfl.standings`,
`nfl.schedules`, `nfl.games`, `nfl.game_team_stats`) and deletes their data and checkpoint directories,
including the pre-`espn/` layout. `nfl.injury_news` is not touched.

Run once after deploying the schema change, then run each `ingest_game_*` notebook to rebuild from the raw `espn/` blobs.
Set the `confirm` widget to `RESET` to proceed.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from lib import schemas

dbutils.widgets.text("confirm", "", "Type RESET to drop ESPN silver tables")
if dbutils.widgets.get("confirm") != "RESET":
    dbutils.notebook.exit("Skipped: set confirm=RESET to run")

ACCOUNT = "footballmanagerli"
ROOT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net"

# Current layout plus the paths used before the espn/ raw-landing change.
LEGACY_DATA = [
    "teams/teams",
    "players/players",
    "rosters/rosters",
    "standings/standings",
    "schedules/schedules",
    "games/games",
    "game-team-stats/game_team_stats",
]
paths = (
    [f"{ROOT}/silver/{name}" for name in schemas.ESPN_TABLES]
    + [f"{ROOT}/_checkpoints/silver_{name}" for name in schemas.ESPN_TABLES]
    + [f"{ROOT}/_checkpoints/{name}" for name in schemas.ESPN_TABLES]
    + [f"{ROOT}/{p}" for p in LEGACY_DATA]
)

for name in schemas.ESPN_TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {schemas.TABLES[name].full_name}")
    print(f"dropped {schemas.TABLES[name].full_name}")

for path in paths:
    try:
        dbutils.fs.rm(path, True)
        print(f"removed {path}")
    except Exception as e:  # noqa: BLE001 - missing paths are fine
        print(f"skip {path}: {type(e).__name__}")